# Notebook 8: Número de victimizaciones de sexo femenino de 16 años o más de edad que han sufrido violencia sexual fuera del ámbito de la pareja

Este dataset es un conjunto de diversas series del INE. La única desagragación que nos interesa es por año.

Nota: una buena forma de buscar estos datasets es escribiendo en Google "INE código" doonde el código es algo parecido a "ODS64".

## Visualizaciones objetivo

Todas son del proyecto de Santa Coloma, a pesar de no haber desagregación territorial.

- [dxvxm-santa_coloma_g-v_3_2_2_1-esp](https://app.flourish.studio/visualisation/13537239/edit), ([-cat](https://app.flourish.studio/visualisation/13253310/edit)): area chart de los diferentes tipos de espacios

- [dxvxm-santa_coloma_g-v_3_2_2_2-esp](https://app.flourish.studio/visualisation/13537287/edit), ([-cat](https://app.flourish.studio/visualisation/13253176/edit)): line chart de agresiones en espacio público vs bar chart de agresiones totales

- [dxvxm-santa_coloma_g-v_3_2_2_3-esp](https://app.flourish.studio/visualisation/13537313/edit), ([-cat](https://app.flourish.studio/visualisation/13253328/edit)): donut chart de agresiones sexuales en espacios públicos vs privados; se puede seleccionar el año a visualizar

In [1]:
import pandas as pd

from pathlib import Path

PATH_DATA = Path("./data")

## Descarga automática por API

El ChatGPT ha generado, tras un par de iteraciones e indicándole el formato, el dataset.

In [12]:
# TODO: desmontar para que, si falla la API, se puedan seguir las instrucciones de descarga manual.

In [ ]:
# Series de interés
series: dict[str, str] = {
    "ODS64": "espacios_abiertos",
    "ODS65": "establecimientos",
    "ODS66": "instalaciones",
    "ODS67": "medios_transporte",
    "ODS68": "dependencias_comunes",
    "ODS69": "recintos",
    "ODS70": "vias_comunicacion",
    "ODS71": "viviendas"
}

# Descargar y preparar cada serie
dataframes = []

for codigo, nombre in series.items():

    url = (
        "https://www.ine.es/consul/excelCesta.do"
        f"?tipofichero=3&serie={codigo}"
    )

    df = pd.read_csv(
        url,
        sep=";",
        encoding="utf-8",
        dtype=str,
        keep_default_na=False
    )

    # Conservar únicamente año y valor
    df = df[["PERIODO", "VALOR"]].copy()

    # Renombrar columnas
    df.columns = ["anyo", nombre]

    # Convertir año a entero
    df["anyo"] = pd.to_numeric(df["anyo"], errors="raise").astype(int)

    # Quitar puntos de los números (1.907 = 1907)
    df[nombre] = (
        df[nombre]
        .str.strip()
        .replace("", pd.NA)
        .str.replace(".", "", regex=False)
    )

    df[nombre] = pd.to_numeric(df[nombre], errors="raise")

    # Comprobar que no hay años duplicados
    if df["anyo"].duplicated().any():
        raise ValueError(f"Años duplicados en {codigo}")

    # Indexar por año para concatenación horizontal
    df = df.set_index("anyo")

    dataframes.append(df)


# Comprobar que todas las series tienen los mismos años
anios_referencia = set(dataframes[0].index)

for df in dataframes[1:]:
    if set(df.index) != anios_referencia:
        raise ValueError("Las series no tienen los mismos años")


# Concatenar horizontalmente
dataset = pd.concat(dataframes, axis=1)

# Ordenar cronológicamente
dataset = dataset.sort_index(ascending=True)

# Convertir el índice en columna
dataset = dataset.reset_index()

# Exportar
# dataset.to_csv(
#     "violencia_sexual_lugar.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

print(dataset.info())

dataset.head(30)

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   anyo                  10 non-null     int64
 1   espacios_abiertos     10 non-null     int64
 2   establecimientos      10 non-null     int64
 3   instalaciones         10 non-null     int64
 4   medios_transporte     10 non-null     int64
 5   dependencias_comunes  10 non-null     int64
 6   recintos              10 non-null     int64
 7   vias_comunicacion     10 non-null     int64
 8   viviendas             10 non-null     int64
dtypes: int64(9)
memory usage: 852.0 bytes
None


,anyo,espacios_abiertos,establecimientos,instalaciones,medios_transporte,dependencias_comunes,recintos,vias_comunicacion,viviendas
0,2015,316,459,363,117,237,11,1156,1448
1,2016,374,584,456,140,234,7,1293,1617
2,2017,375,718,486,199,238,16,1474,1790
3,2018,405,933,750,246,278,12,1721,2251
4,2019,502,1048,828,264,285,11,1827,2641
5,2020,368,612,666,140,201,10,1116,2633
6,2021,518,940,884,176,258,16,1712,3660
7,2022,658,1385,1144,246,240,19,2117,3676
8,2023,717,1757,1466,291,305,23,2273,4175
9,2024,728,1907,1565,360,313,22,2632,4353


Los valores son el número de víctimas. Cada columna es un tipo de espacio.